# OPI 使用入门：第一个示例

本教程演示如何将 ORCA 与 ORCA Python 接口（OPI）配合使用。以水分子能量计算为例，编写脚本执行 OPI 计算并访问结果；这两项操作均可通过 OPI 完成。

本 Notebook 将：
1. 导入所需 Python 依赖。
2. 定义工作目录。
3. 设置输入结构。
4. 定义计算设置。
5. 执行 ORCA 计算。
6. 检查正常终止与收敛。
7. 处理结果。

## 第 1 步： 安装第三方依赖

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade py3Dmol

## 第 2 步：导入依赖

使用 OPI 时，首先导入所需模块。此外，还导入 `py3Dmol` 等可视化模块。可能需要将 `py3Dmol` 安装到 OPI 的 `venv` 中，例如激活 `.venv` 后运行 `uv pip install py3Dmol`。

In [1]:
from pathlib import Path
import shutil

# > OPI imports for performing ORCA calculations and reading output
from opi.core import Calculator
from opi.output.core import Output
from opi.input.simple_keywords import Dft, Task
from opi.input.structures.structure import Structure

# > for visualization of molecules
import py3Dmol

## 第 3 步：工作目录

使用 OPI 时，建议定义一个子目录，用于执行实际的 ORCA 计算。这里创建 `how_to_opi` 目录。

In [2]:
# > Calculation is performed in `how_to_opi`
working_dir = Path("how_to_opi")
# > The `working_dir`is automatically (re-)created
shutil.rmtree(working_dir, ignore_errors=True)
working_dir.mkdir()

## 第 4 步： 设置输入结构

以埃为单位定义结构的笛卡尔坐标，并将其可视化：


In [3]:
# > Define cartesian coordinates in angstrom as python string 
xyz_data = """\
3

O      0.00000   -0.00000    0.00000
H      0.00000    0.96899    0.00000
H      0.93966   -0.23409    0.03434\n
"""
# > Visualize the input structure
view = py3Dmol.view(width=400, height=400)
view.addModel(xyz_data, 'xyz')
view.setStyle({}, {'stick': {}, 'sphere': {'scale': 0.3}})
view.zoomTo()
view.show()

# > Write the input structure to a file
with open(working_dir / "struc.xyz","w") as f:
    f.write(xyz_data)
# > Read structure into object
structure = Structure.from_xyz(working_dir / "struc.xyz")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 第 5 步： 定义计算设置
接下来定义 ORCA 优化使用的设置。可以创建一个 calculator 对象，并向其添加简单关键词：

In [4]:
def setup_calc(basename : str, working_dir: Path, structure: Structure, ncores: int = 1) -> Calculator:
    # > Set up a Calculator object, the basename for the ORCA calculation is also set
    calc = Calculator(basename=basename, working_dir=working_dir)
    # > Assign structure to calculator
    calc.structure = structure

    # Define a level of theory: we use r2SCAN-3c for geometry optimization
    sk_list = [
        Dft.R2SCAN_3C, # > r2SCAN-3c method (Comes with a predefined basis set)
        Task.SP # > Perform the singlepoint
    ]

    # > Use simple keywords in calculator
    calc.input.add_simple_keywords(*sk_list)

    # > Define number of CPUs for the calcualtion
    calc.input.ncores = ncores # > CPUs for this ORCA run (default: 1)

    return calc

calc = setup_calc("single_point", working_dir=working_dir, structure=structure)

## 第 6 步： 执行计算

执行单点计算时，先将输入写成 ORCA 格式，再使用 calculator 类的 `run()` 函数运行作业。可通过其 `get_output()` 函数将 ORCA 输出写入 calculator 对象。

In [5]:
def run_calc(calc: Calculator) -> Output:
    # > Write the ORCA input file
    calc.write_input()
    # > Run the ORCA calculation
    print("Running ORCA calculation ...", end="")
    calc.run()
    print("   Done")

    # > Get the output object
    output = calc.get_output()
    
    return output

output = run_calc(calc)

Running ORCA calculation ...   Done


## 第 7 步： 检查正常终止与收敛
执行自动化计算时，应始终检查是否正常终止和收敛。简单检查示例如下：

In [6]:
def check_and_parse_output(output: Output):
    # > Check for proper termination of ORCA
    status = output.terminated_normally()
    if not status:
        # > ORCA did not terminate normally
        raise RuntimeError("ORCA did not terminate normally. Please check RUN/single_point.out")
    else:
        # > ORCA did terminate normally so we can parse the output
        output.parse()

    # Now check for convergence of the SCF
    if output.results_properties.geometries[0].single_point_data.converged:
        print("SCF CONVERGED")
    else:
        raise RuntimeError("SCF DID NOT CONVERGE")
    
check_and_parse_output(output)

SCF CONVERGED


## 第 8 步： 访问结果
计算成功后，可以访问所得性质。通常可通过 `output` 对象获取：

In [7]:
# > Total energy in Hartee
Etot = output.results_properties.geometries[0].single_point_data.finalenergy

# > Print total energy
print("Final energy in Hartee: ",Etot)

Final energy in Hartee:  -76.41884216440265


## 小结

本 Notebook 演示了如何使用 ORCA Python 接口（OPI）执行 DFT 单点能计算。